# SQL practice 5: subqueries, CTEs and set operations

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-practice-05-subqueries-ctes-and-set-operations.ipynb)

14 exercises on scalar, correlated and EXISTS subqueries, CTEs, hierarchies and set operations. Each exercise names the [reference entries](sql-reference-05-subqueries-ctes-and-set-operations.ipynb) it practises, shows the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and the helpers from [SQL reference 5](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Exercises

7 easy, 5 medium, 2 hard. Write your query in the empty cell under each exercise, run it with `show(q)` or `spark.sql(q).show()`, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Orders above the overall average](#exercise-1-orders-above-the-overall-average) | Easy | [Scalar subqueries](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#scalar-subqueries) |
| 2 | [Orders above the customer's own average](#exercise-2-orders-above-the-customers-own-average) | Medium | [Scalar subqueries](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#scalar-subqueries) |
| 3 | [Customers above the average customer](#exercise-3-customers-above-the-average-customer) | Medium | [`WITH` clauses](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#with-clauses) |
| 4 | [Products never in a paid order](#exercise-4-products-never-in-a-paid-order) | Easy | [`IN` and `EXISTS` subqueries](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#in-and-exists-subqueries) |
| 5 | [Customers who bought a category](#exercise-5-customers-who-bought-a-category) | Easy | [`IN` and `EXISTS` subqueries](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#in-and-exists-subqueries) |
| 6 | [Latest order per customer](#exercise-6-latest-order-per-customer) | Medium | [Scalar subqueries](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#scalar-subqueries) |
| 7 | [A unified activity feed](#exercise-7-a-unified-activity-feed) | Easy | [`UNION`, `INTERSECT` and `EXCEPT`](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#union-intersect-and-except) |
| 8 | [Every SKU referenced anywhere](#exercise-8-every-sku-referenced-anywhere) | Easy | [`UNION`, `INTERSECT` and `EXCEPT`](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#union-intersect-and-except) |
| 9 | [Customers with paid and shipped orders](#exercise-9-customers-with-paid-and-shipped-orders) | Easy | [`UNION`, `INTERSECT` and `EXCEPT`](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#union-intersect-and-except) |
| 10 | [Sold SKUs missing from the catalogue](#exercise-10-sold-skus-missing-from-the-catalogue) | Easy | [`UNION`, `INTERSECT` and `EXCEPT`](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#union-intersect-and-except) |
| 11 | [Diff two snapshots](#exercise-11-diff-two-snapshots) | Medium | [Comparing two tables](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#comparing-two-tables) |
| 12 | [New customers per day](#exercise-12-new-customers-per-day) | Medium | [`WITH` clauses](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#with-clauses) |
| 13 | [Management chain without recursion](#exercise-13-management-chain-without-recursion) | Hard | [Recursive queries and their Spark alternatives](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#recursive-queries-and-their-spark-alternatives) |
| 14 | [Second-highest salary per department, no windows](#exercise-14-second-highest-salary-per-department-no-windows) | Hard | [Derived tables](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#derived-tables) |

## Exercise 1: Orders above the overall average

**Difficulty:** Easy · about 3 min · **Practises:** [Scalar subqueries](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#scalar-subqueries)

*Uncorrelated scalar subqueries compute a benchmark once.*

Return `order_id` and `amount` of orders above the average order amount, ordered by `amount` descending.

**Expected output**

| order_id | amount |
| --- | --- |
| 9 | 1180.00 |
| 7 | 310.00 |
| 4 | 260.00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT order_id, amount
    FROM orders
    WHERE amount > (SELECT avg(amount) FROM orders)
    ORDER BY amount DESC
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 2: Orders above the customer's own average

**Difficulty:** Medium · about 10 min · **Practises:** [Scalar subqueries](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#scalar-subqueries)

*Correlated subqueries compare each row with its own group.*

Return `order_id`, `customer_id` and `amount` of orders larger than that customer's average order amount, ordered by `order_id`.

**Expected output**

| order_id | customer_id | amount |
| --- | --- | --- |
| 1 | 1 | 120.00 |
| 2 | 1 | 84.50 |
| 4 | 2 | 260.00 |
| 7 | 3 | 310.00 |
| 9 | 4 | 1180.00 |
| 11 | 6 | 210.00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT o.order_id, o.customer_id, o.amount
    FROM orders o
    WHERE o.amount > (SELECT avg(i.amount) FROM orders i WHERE i.customer_id = o.customer_id)
    ORDER BY o.order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 3: Customers above the average customer

**Difficulty:** Medium · about 10 min · **Practises:** [`WITH` clauses](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#with-clauses)

*Two-step metrics read best as named steps.*

Return `name` and `total` (sum of non-returned order amounts) for customers whose total is above the average of those totals, ordered by `total` descending.

**Expected output**

| name | total |
| --- | --- |
| Delta Grocers | 1180.00 |
| Cyan Traders | 459.00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    WITH totals AS (
        SELECT customer_id, sum(amount) AS total
        FROM orders WHERE status <> 'returned'
        GROUP BY customer_id
    )
    SELECT c.name, t.total
    FROM totals t
    JOIN customers c ON c.customer_id = t.customer_id
    WHERE t.total > (SELECT avg(total) FROM totals)
    ORDER BY t.total DESC
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 4: Products never in a paid order

**Difficulty:** Easy · about 5 min · **Practises:** [`IN` and `EXISTS` subqueries](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#in-and-exists-subqueries)

*Dead-stock style reports: what has never converted into revenue?*

Return `sku` and `category` of products that never appear on an order with status `paid`, ordered by `sku`, using `NOT EXISTS`.

**Expected output**

| sku | category |
| --- | --- |
| SKU-4002 | Kitchen |
| SKU-4005 | Pantry |
| SKU-4009 | Pantry |
| SKU-4010 | Garden |
| SKU-4011 | Pantry |
| SKU-4013 | Outdoor |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT p.sku, p.category
    FROM products p
    WHERE NOT EXISTS (
        SELECT 1
        FROM order_items i
        JOIN orders o ON o.order_id = i.order_id
        WHERE i.sku = p.sku AND o.status = 'paid'
    )
    ORDER BY p.sku
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 5: Customers who bought a category

**Difficulty:** Easy · about 5 min · **Practises:** [`IN` and `EXISTS` subqueries](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#in-and-exists-subqueries)

*Filtering by a property of related rows without duplicating the outer rows.*

Return `customer_id` and `name` of customers who ordered at least one Kitchen product, ordered by `customer_id`.

**Expected output**

| customer_id | name |
| --- | --- |
| 1 | Acme Corp |
| 5 | Echo Supplies |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id, name
    FROM customers
    WHERE customer_id IN (
        SELECT o.customer_id
        FROM orders o
        JOIN order_items i ON i.order_id = o.order_id
        JOIN products p ON p.sku = i.sku
        WHERE p.category = 'Kitchen'
    )
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 6: Latest order per customer

**Difficulty:** Medium · about 10 min · **Practises:** [Scalar subqueries](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#scalar-subqueries)

*"Latest row per key" without window functions, a common interview restriction.*

Return `customer_id`, `order_id` and `order_ts` of each customer's most recent order, ordered by `customer_id`.

**Expected output**

| customer_id | order_id | order_ts |
| --- | --- | --- |
| 1 | 3 | 2026-01-05 11:40:00 |
| 2 | 5 | 2026-01-07 08:05:00 |
| 3 | 7 | 2026-01-09 12:10:00 |
| 4 | 9 | 2026-01-12 08:55:00 |
| 5 | 10 | 2026-01-12 13:00:00 |
| 6 | 12 | 2026-01-14 00:30:00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT o.customer_id, o.order_id, o.order_ts
    FROM orders o
    WHERE o.order_ts = (SELECT max(i.order_ts) FROM orders i WHERE i.customer_id = o.customer_id)
    ORDER BY o.customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 7: A unified activity feed

**Difficulty:** Easy · about 5 min · **Practises:** [`UNION`, `INTERSECT` and `EXCEPT`](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#union-intersect-and-except)

*Stacking events from different tables into one timeline.*

Return `customer_id`, `kind` (`signup` or `order`) and `at` (a timestamp) for customer 3's signup and orders, ordered by `at`.

**Expected output**

| customer_id | kind | at |
| --- | --- | --- |
| 3 | signup | 2024-03-15 00:00:00 |
| 3 | order | 2026-01-07 17:45:00 |
| 3 | order | 2026-01-09 12:10:00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id, 'signup' AS kind, CAST(signup_date AS TIMESTAMP) AS at
    FROM customers WHERE customer_id = 3
    UNION ALL
    SELECT customer_id, 'order' AS kind, order_ts AS at
    FROM orders WHERE customer_id = 3
    ORDER BY at
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 8: Every SKU referenced anywhere

**Difficulty:** Easy · about 3 min · **Practises:** [`UNION`, `INTERSECT` and `EXCEPT`](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#union-intersect-and-except)

*UNION removes duplicates; UNION ALL does not.*

Return one row with `distinct_skus` (SKUs that appear in `products` or `order_items`, counted once) and `stacked_rows` (the row count of the same two lists stacked with duplicates).

**Expected output**

| distinct_skus | stacked_rows |
| --- | --- |
| 14 | 30 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT (SELECT count(*) FROM (SELECT sku FROM products UNION SELECT sku FROM order_items)) AS distinct_skus,
           (SELECT count(*) FROM (SELECT sku FROM products UNION ALL SELECT sku FROM order_items)) AS stacked_rows
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 9: Customers with paid and shipped orders

**Difficulty:** Easy · about 5 min · **Practises:** [`UNION`, `INTERSECT` and `EXCEPT`](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#union-intersect-and-except)

*INTERSECT answers "in both sets".*

Return the `customer_id`s that have at least one paid order and at least one shipped order, ordered.

**Expected output**

| customer_id |
| --- |
| 1 |
| 3 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id FROM orders WHERE status = 'paid'
    INTERSECT
    SELECT customer_id FROM orders WHERE status = 'shipped'
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 10: Sold SKUs missing from the catalogue

**Difficulty:** Easy · about 3 min · **Practises:** [`UNION`, `INTERSECT` and `EXCEPT`](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#union-intersect-and-except)

*EXCEPT is a compact anti join on whole rows; here it finds referential-integrity gaps.*

Return the `sku`s that appear in `order_items` but not in `products`, ordered, using `EXCEPT`.

**Expected output**

| sku |
| --- |
| SKU-4014 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT sku FROM order_items
    EXCEPT
    SELECT sku FROM products
    ORDER BY sku
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 11: Diff two snapshots

**Difficulty:** Medium · about 10 min · **Practises:** [Comparing two tables](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#comparing-two-tables)

*Regression-testing a pipeline change by diffing its output with the previous run.*

Using `snapshot_old` and `snapshot_new` below, return `side` (`only_old` or `only_new`), `id` and `email` for rows that are not identical in both, ordered by `id`, then `side`.

In [ ]:
spark.sql("CREATE OR REPLACE TEMP VIEW snapshot_old AS SELECT * FROM VALUES (1, 'a@x'), (2, 'b@x'), (3, NULL) AS t(id, email)")
spark.sql("CREATE OR REPLACE TEMP VIEW snapshot_new AS SELECT * FROM VALUES (1, 'a@x'), (2, 'b@new'), (3, NULL), (4, 'd@x') AS t(id, email)")

**Expected output**

| side | id | email |
| --- | --- | --- |
| only_new | 2 | b@new |
| only_old | 2 | b@x |
| only_new | 4 | d@x |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT 'only_old' AS side, id, email FROM (SELECT * FROM snapshot_old EXCEPT SELECT * FROM snapshot_new)
    UNION ALL
    SELECT 'only_new' AS side, id, email FROM (SELECT * FROM snapshot_new EXCEPT SELECT * FROM snapshot_old)
    ORDER BY id, side
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 12: New customers per day

**Difficulty:** Medium · about 10 min · **Practises:** [`WITH` clauses](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#with-clauses)

*First-touch metrics need the first event per entity, then a count per day.*

Return `day` (date of a customer's first order) and `new_customers`, ordered by `day`.

**Expected output**

| day | new_customers |
| --- | --- |
| 2026-01-05 | 1 |
| 2026-01-06 | 1 |
| 2026-01-07 | 1 |
| 2026-01-10 | 1 |
| 2026-01-12 | 1 |
| 2026-01-13 | 1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    WITH first_orders AS (
        SELECT customer_id, min(order_ts) AS first_ts
        FROM orders
        GROUP BY customer_id
    )
    SELECT to_date(first_ts) AS day, count(*) AS new_customers
    FROM first_orders
    GROUP BY to_date(first_ts)
    ORDER BY day
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 13: Management chain without recursion

**Difficulty:** Hard · about 15 min · **Practises:** [Recursive queries and their Spark alternatives](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#recursive-queries-and-their-spark-alternatives)

*Spark 4.0 has no recursive CTEs, so known-depth hierarchies use repeated self joins.*

Return `name` and `chain`: the employee's name followed by their managers up to the top, joined with ` > ` (for example `Dana > Brett > Ada`). The hierarchy is at most three levels deep. Order by `name`.

**Expected output**

| name | chain |
| --- | --- |
| Ada | Ada |
| Brett | Brett > Ada |
| Chen | Chen > Ada |
| Dana | Dana > Brett > Ada |
| Eli | Eli > Ada |
| Farah | Farah > Eli > Ada |
| Gus | Gus > Eli > Ada |
| Hana | Hana > Ada |
| Ivan | Ivan > Hana > Ada |
| Jo | Jo > Hana > Ada |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT e.name,
           concat_ws(' > ', e.name, m1.name, m2.name, m3.name) AS chain
    FROM employees e
    LEFT JOIN employees m1 ON m1.emp_id = e.manager_id
    LEFT JOIN employees m2 ON m2.emp_id = m1.manager_id
    LEFT JOIN employees m3 ON m3.emp_id = m2.manager_id
    ORDER BY e.name
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 14: Second-highest salary per department, no windows

**Difficulty:** Hard · about 15 min · **Practises:** [Derived tables](sql-reference-05-subqueries-ctes-and-set-operations.ipynb#derived-tables)

*An interview classic when window functions are off the table.*

Return `dept` and `second_highest` (the second-highest **distinct** salary in the department, NULL when there is none) for every department that has a name, ordered by `dept`.

**Expected output**

| dept | second_highest |
| --- | --- |
| Engineering | 155000 |
| Finance | 88000 |
| Sales | 95000 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT d.dept, max(e.salary) AS second_highest
    FROM (SELECT dept, max(salary) AS top_salary
          FROM employees WHERE dept IS NOT NULL GROUP BY dept) d
    LEFT JOIN employees e ON e.dept = d.dept AND e.salary < d.top_salary
    GROUP BY d.dept
    ORDER BY d.dept
"""
result = spark.sql(q)
result.show(truncate=False)

**Next:** look anything up in [SQL reference 5: subqueries, CTEs and set operations](sql-reference-05-subqueries-ctes-and-set-operations.ipynb), or try mixed interview problems in [SQL interview drills](sql-13-interview-drills.ipynb).